In [1]:

from enum import Enum
from pydantic import BaseModel, Field, model_validator, ValidationInfo, validator
from typing import Optional, List
import re

class Fact(BaseModel):
    fact: str = Field(...)
    substring_quote: List[str] = Field(...)

    @model_validator(mode="after")
    def validate_sources(self, info: ValidationInfo) -> "Fact":
        text_chunks = info.context.get("text_chunk", None)
        spans = list(self.get_spans(text_chunks))
        self.substring_quote = [text_chunks[span[0] : span[1]] for span in spans]
        return self

    def get_spans(self, context):
        for quote in self.substring_quote:
            yield from self._get_span(quote, context)

    def _get_span(self, quote, context):
        for match in re.finditer(re.escape(quote), context):
            yield match.span()

class Bm_recent_blast_method(Enum):
    MORPHOLOGY = "Morphology"
    FLOWCYTOMETRY = "Flow cytometry"

class Cns_status(Enum):
    CNS1 = "CNS 1"
    CNS2 = "CNS 2"
    CNS3 = "CNS 3"

class YesNo(Enum):
    YES = "Yes"
    NO = "No"

class Entity0(BaseModel):
    '''
    Extracted from the clinical trial eligibility document.
Example: the answer is >=2 if "Second or greater relapse"; the answer is 1 if "first relapses"; 
    '''
    count_relapse_exist: YesNo = Field(..., description="Does the eligibility criteria mention the number of relapses of the patient?")
    count_relapse: Optional[str] = Field(
        None, 
        description="If so, what is the minimum number of relapses required to be included? ")
    answer_facts: List[Fact] = Field(..., description="Exact fact quotes from the note supporting the answer.")

    @model_validator(mode="after")
    def skip_if_not_exist(self, info: ValidationInfo) -> "Entity0":
        if getattr(self, "count_relapse_exist") == YesNo.NO:
            if getattr(self, "count_relapse") is not None:
                raise ValueError("count_relapse must be None if count_relapse_exist is 'NO'")
            setattr(self, "count_relapse", None)
            self.answer_facts = []
        return self

  
    @model_validator(mode='after')
    def validate_sources(self) -> "Entity0":
        self.answer_facts = [fact for fact in self.answer_facts if len(fact.substring_quote) > 0]
        return self



In [2]:
import instructor
from enum import Enum
from openai import AzureOpenAI
from functools import partial
import pandas as pd
from datetime import datetime
from typing import TypeVar, Type, Optional, List
from pydantic import BaseModel, Field
import os
import re
from itertools import islice


In [3]:
# Define your Azure OpenAI configuration
model = "gpt4o_cursor"
engine = "gpt4o_cursor"
endpoint = "https://gpt4o-xj.openai.azure.com/"

api_version = "2024-02-15-preview"
api_type = "azure"

# key_name = "AZURE_KEY"
api_key = 'YOUR-API-KEY'

# Create the Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    api_key=api_key,
    azure_endpoint=endpoint,
    azure_deployment=engine
)

# Patch the client with instructor
client = instructor.patch(client)

# Optionally, create a partial function with default parameters
in_client = partial(
    client.chat.completions.create,
    model=model,
    max_retries=2  # Retries in case of API issues
)


In [4]:
input_dir = 'input_test/'
df = pd.DataFrame(columns=['File Name', 'Inclusion Criteria', 'Exclusion Criteria'])
abstract_list = [] 

for file_name in os.listdir(input_dir):
    if file_name.endswith('.txt'):
        file_path = os.path.join(input_dir, file_name)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            ### Split the notes to inclusion and exclusion criteria seperately
            inclusion_criteria = content.split('Exclusion Criteria:')[0].split('Inclusion Criteria:')[-1].strip()
            exclusion_criteria = content.split('Exclusion Criteria:')[-1].strip()
            new_row = pd.DataFrame({'File Name': [file_name], 'Inclusion Criteria': [inclusion_criteria], 'Exclusion Criteria': [exclusion_criteria]})
            df = pd.concat([df, new_row], ignore_index=True)

abstract_list = df[['File Name', 'Inclusion Criteria', 'Exclusion Criteria']].values.tolist()
print("Abstracts have been successfully read into a list.")

Abstracts have been successfully read into a list.


In [5]:
# For testing purposes
# def ask_ai(criteria:str, type:str) -> T:
#     return in_client(
#         response_model=Entity0,
#         temperature=0,
#         messages=[
#             {
#                 "role": "system",
#                 "content": "You will help extract entities from clinical protocol documents",
#             },
#             {
#                 "role": "user",
#                 "content": f"Extract information from the clinical trial eligibility document ({type} criteria).",
#             },
#             {
#                 "role": "user",
#                 "content": f"Here is the {type} Criteria section {criteria}",
#             },
#         ],
#         validation_context={"text_chunk": criteria},
#     ) 
# for abstract in abstract_list:
#     file_name = abstract[0]
#     clean_text_ic = abstract[1]
#     clean_text_ec = abstract[2]

#     output_ic = ask_ai(criteria=clean_text_ic, type='Inclusion')
#     print(output_ic)

In [6]:
T = TypeVar("T", bound=BaseModel)

class LLMEntityExtractor:
    def __init__(self, response_model: Type[T], client):
        self.response_model = response_model
        self.client = client

    def ask_ai(self, criteria:str, type:str) -> T:
        return self.client(
            response_model=self.response_model,
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": "You will help extract entities from clinical protocol documents",
                },
                {
                    "role": "user",
                    "content": f"Extract information from the clinical trial eligibility document ({type} criteria).",
                },
                {
                    "role": "user",
                    "content": f"Here is the {type} Criteria section {criteria}",
                },
            ],
            validation_context={"text_chunk": criteria},
        ) 

    def output(self, criteria: str, type: str):
        try:
            response = self.ask_ai(criteria, type)
            entity_dict = response.dict()

            for key, value in entity_dict.items():
                if isinstance(value, list):
                    if value and isinstance(value[0], Enum):
                        entity_dict[key] = [v.value for v in value]  # Extract the string representation
                    # else: keep the list as is
                elif isinstance(value, Enum):
                    entity_dict[key] = value.value  # For single Enum fields

            entity_value_data = {
                "Entity": list(entity_dict.keys()),
                "Value": list(entity_dict.values()),
            }
            df = pd.DataFrame(entity_value_data)
            return df
        
        except Exception as e:
            print(f"Error in output method: {str(e)}")
            # Return an empty DataFrame if an error occurs
            return pd.DataFrame(columns=["Entity", "Value"])

In [7]:
import json

for abstract in abstract_list:
    file_name = abstract[0]
    clean_text_ic = abstract[1]
    clean_text_ec = abstract[2]

    all_outputs = []  # List to store outputs for all models

    for Model in [Entity0]: #, Entity1, Entity2, Entity3, Entity4, Entity5, Entity6]:
        Classifier_IC = LLMEntityExtractor(response_model=Model, client=in_client)
        output_ic = Classifier_IC.output(criteria=clean_text_ic, type='Inclusion')

        all_outputs.extend(output_ic.to_dict(orient="records"))

    # Save all outputs to a single JSON file
    output_file_path = f"{file_name}_inclusion.json"
    with open(output_file_path, "w") as json_file:
        json.dump(all_outputs, json_file, indent=4)

    print(f"Saved all inclusion outputs for {file_name} to {output_file_path}")

Saved all inclusion outputs for NCT04915612.txt to NCT04915612.txt_inclusion.json


/var/folders/vh/jcnnz97d2bl8hm9yqsltgx2h0000gn/T/ipykernel_79649/2982577277.py:32: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  entity_dict = response.dict()


In [1]:
run = [{'count_relapse_exist': 'Yes', 'count_relapse': '1', 'answer_facts': [{'fact': 'Relapsed refractory acute myeloid leukemia by World Health Organization (WHO) criteria', 'substring_quote': ['Relapsed refractory acute myeloid leukemia by World Health Organization (WHO) criteria']}]}, {'bm_recent_blast_pct_meas_exist': 'Yes', 'bm_recent_blast_pct_meas': 5, 'answer_facts': [{'fact': 'Patients must have >= 5% blasts in the bone marrow as assessed by morphology or flow cytometry.', 'substring_quote': ['>= 5% blasts in the bone marrow']}, {'fact': 'patients may be enrolled if there is unequivocal evidence of leukemia with >= 5% blasts in the peripheral blood', 'substring_quote': ['>= 5% blasts in the peripheral blood']}]}, {'bm_recent_blast_method_exist': 'Yes', 'bm_recent_blast_method': ['Morphology', 'Flow cytometry'], 'answer_facts': [{'fact': 'Patients must have >= 5% blasts in the bone marrow as assessed by morphology or flow cytometry.', 'substring_quote': ['morphology', 'flow cytometry']}]}, {'periph_blood_blast_abs_meas_exist': 'Yes', 'periph_blood_blast_abs_meas': 5, 'answer_facts': [{'fact': 'Patients must have >= 5% blasts in the bone marrow as assessed by morphology or flow cytometry. However, if an adequate bone marrow sample cannot be obtained, patients may be enrolled if there is unequivocal evidence of leukemia with >= 5% blasts in the peripheral blood', 'substring_quote': ['>= 5% blasts in the peripheral blood']}]}, {'periph_blood_blast_pct_meas_exist': 'Yes', 'periph_blood_blast_pct_meas': 5, 'answer_facts': [{'fact': '>= 5% blasts in the peripheral blood', 'substring_quote': ['>= 5% blasts in the peripheral blood']}]}, {'corneal_epitheliopathy_exist': 'No', 'corneal_epitheliopathy': None, 'answer_facts': []}, {'cns_status_exist': 'No', 'cns_status': None, 'answer_facts': []}]
[{'count_relapse_exist': 'No', 'count_relapse': None, 'answer_facts': []}, {'bm_recent_blast_pct_meas_exist': 'No', 'bm_recent_blast_pct_meas': None, 'answer_facts': []}, {'bm_recent_blast_method_exist': 'No', 'bm_recent_blast_method': None, 'answer_facts': []}, {'periph_blood_blast_abs_meas_exist': 'No', 'periph_blood_blast_abs_meas': None, 'answer_facts': []}, {'periph_blood_blast_pct_meas_exist': 'No', 'periph_blood_blast_pct_meas': None, 'answer_facts': []}, {'corneal_epitheliopathy_exist': 'No', 'corneal_epitheliopathy': None, 'answer_facts': []}, {'cns_status_exist': 'Yes', 'cns_status': ['CNS 1', 'CNS 2', 'CNS 3'], 'answer_facts': [{'fact': 'Presence of clinically significant uncontrolled central nervous system (CNS) pathology such as epilepsy, childhood seizure, paresis, aphasia, stroke, severe brain injuries, organic brain syndrome, or psychosis', 'substring_quote': ['clinically significant uncontrolled central nervous system (CNS) pathology']}, {'fact': 'Evidence of active cerebral/meningeal disease', 'substring_quote': ['active cerebral/meningeal disease']}, {'fact': 'Patients may have history of CNS leukemic involvement if definitively treated with prior therapy and no evidence of active disease at the time of consent with at least 2 consecutive spinal fluid negative assessments for residual leukemia and negative imaging', 'substring_quote': ['history of CNS leukemic involvement', 'no evidence of active disease', '2 consecutive spinal fluid negative assessments for residual leukemia', 'negative imaging']}]}]

[{'count_relapse_exist': 'No', 'count_relapse': None, 'answer_facts': []},
 {'bm_recent_blast_pct_meas_exist': 'No',
  'bm_recent_blast_pct_meas': None,
  'answer_facts': []},
 {'bm_recent_blast_method_exist': 'No',
  'bm_recent_blast_method': None,
  'answer_facts': []},
 {'periph_blood_blast_abs_meas_exist': 'No',
  'periph_blood_blast_abs_meas': None,
  'answer_facts': []},
 {'periph_blood_blast_pct_meas_exist': 'No',
  'periph_blood_blast_pct_meas': None,
  'answer_facts': []},
 {'corneal_epitheliopathy_exist': 'No',
  'corneal_epitheliopathy': None,
  'answer_facts': []},
 {'cns_status_exist': 'Yes',
  'cns_status': ['CNS 1', 'CNS 2', 'CNS 3'],
  'answer_facts': [{'fact': 'Presence of clinically significant uncontrolled central nervous system (CNS) pathology such as epilepsy, childhood seizure, paresis, aphasia, stroke, severe brain injuries, organic brain syndrome, or psychosis',
    'substring_quote': ['clinically significant uncontrolled central nervous system (CNS) pathology']

In [4]:
import pandas as pd
final_output_ic = pd.DataFrame(run)

In [5]:
final_output_ic

,count_relapse_exist,count_relapse,answer_facts,bm_recent_blast_pct_meas_exist,bm_recent_blast_pct_meas,bm_recent_blast_method_exist,bm_recent_blast_method,periph_blood_blast_abs_meas_exist,periph_blood_blast_abs_meas,periph_blood_blast_pct_meas_exist,periph_blood_blast_pct_meas,corneal_epitheliopathy_exist,corneal_epitheliopathy,cns_status_exist,cns_status
0,Yes,1,[{'fact': 'Relapsed refractory acute myeloid l...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,[{'fact': 'Patients must have >= 5% blasts in ...,Yes,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,[{'fact': 'Patients must have >= 5% blasts in ...,NaN,NaN,Yes,"[Morphology, Flow cytometry]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,[{'fact': 'Patients must have >= 5% blasts in ...,NaN,NaN,NaN,NaN,Yes,5.0,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,[{'fact': '>= 5% blasts in the peripheral bloo...,NaN,NaN,NaN,NaN,NaN,NaN,Yes,5.0,NaN,NaN,NaN,NaN
5,NaN,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN
6,NaN,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN
